# G-Code XGBoost Classifier: Gun Part Detection

Uses 28 hand-crafted statistical features parsed from g-code movement commands.
No neural network — just gradient-boosted trees.

**Advantages over the embedding approach:**
- Resistant to overfitting on small datasets (~2k samples)
- Fast training (seconds, not hours)
- Fully interpretable (feature importance, SHAP)
- Generalises better to unseen g-code because features encode physics, not memorised sequences


In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'xgboost', 'scikit-learn', 'pandas', 'numpy', 'matplotlib', 'seaborn',
    'datasets', 'tqdm', 'shap'])


In [ ]:
import os
import json
import pickle
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import GroupShuffleSplit, cross_val_score, StratifiedGroupKFold
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve
from sklearn.preprocessing import StandardScaler
from sklearn.calibration import calibration_curve
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from tqdm import tqdm

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)


## 1. Configuration


In [ ]:
HF_DATASET = "jungter/gcode-to-model-gn"
VAL_SPLIT = 0.2
RANDOM_SEED = 42

# XGBoost hyperparameters — tuned for small datasets
XGB_PARAMS = {
    "objective": "binary:logistic",
    "eval_metric": "logloss",
    "max_depth": 4,            # shallow trees = less overfitting
    "learning_rate": 0.05,     # slow learning
    "n_estimators": 300,       # many weak learners
    "subsample": 0.7,          # row sampling per tree
    "colsample_bytree": 0.7,   # feature sampling per tree
    "min_child_weight": 5,     # minimum samples per leaf
    "reg_alpha": 0.5,          # L1 regularisation
    "reg_lambda": 2.0,         # L2 regularisation
    "gamma": 0.1,              # minimum loss reduction for split
    "random_state": RANDOM_SEED,
    "verbosity": 0,
}

print("XGBoost params:")
for k, v in XGB_PARAMS.items():
    print(f"  {k}: {v}")


## 2. Load Dataset from HuggingFace


In [ ]:
from datasets import load_dataset

hf_ds = load_dataset(HF_DATASET, split="train")
print(f"Loaded {len(hf_ds)} samples from {HF_DATASET}")

df = pd.DataFrame({
    "gcode_text": hf_ds["gcode_text"],
    "label": hf_ds["label"],
    "category": hf_ds["category"],
    "part": hf_ds["part"],
    "filename": hf_ds["filename"],
})

print(f"\nLabel distribution:\n{df['label'].value_counts()}")
print(f"\nCategories: {df['category'].unique()}")
print(f"Unique parts: {df['part'].nunique()}")


## 3. G-Code Feature Extraction

Parse each g-code file into a fixed-length numerical feature vector:
- Movement statistics (G0 vs G1, travel vs extrusion ratio, layer count)
- Coordinate statistics (X/Y/Z ranges, centroids, spread)
- Speed/feed rate statistics
- Extrusion statistics


In [ ]:
def extract_gcode_features(gcode_text: str) -> dict:
    """Extract numerical features from g-code movement commands only (no header metadata)."""
    lines = gcode_text.splitlines()
    total_lines = len(lines)

    g0_count = 0
    g1_count = 0
    x_vals, y_vals, z_vals = [], [], []
    e_vals = []
    f_vals = []
    layer_count = 0

    for line in lines:
        stripped = line.strip()
        if not stripped:
            continue
        if stripped.startswith(";"):
            if ";LAYER:" in stripped.upper():
                layer_count += 1
            continue

        if stripped.startswith("G0 ") or stripped.startswith("G0\t"):
            g0_count += 1
        elif stripped.startswith("G1 ") or stripped.startswith("G1\t"):
            g1_count += 1

        parts = stripped.split(";")[0].split()
        for p in parts:
            try:
                if p.startswith("X"): x_vals.append(float(p[1:]))
                elif p.startswith("Y"): y_vals.append(float(p[1:]))
                elif p.startswith("Z"): z_vals.append(float(p[1:]))
                elif p.startswith("E"): e_vals.append(float(p[1:]))
                elif p.startswith("F"): f_vals.append(float(p[1:]))
            except ValueError:
                pass

    def safe_stats(vals):
        if not vals:
            return 0, 0, 0, 0
        arr = np.array(vals)
        return float(arr.min()), float(arr.max()), float(arr.mean()), float(arr.std())

    x_min, x_max, x_mean, x_std = safe_stats(x_vals)
    y_min, y_max, y_mean, y_std = safe_stats(y_vals)
    z_min, z_max, z_mean, z_std = safe_stats(z_vals)
    e_min, e_max, e_mean, e_std = safe_stats(e_vals)
    f_min, f_max, f_mean, f_std = safe_stats(f_vals)

    total_moves = g0_count + g1_count
    extrusion_ratio = g1_count / max(total_moves, 1)

    features = {
        "total_lines": total_lines,
        "layer_count": layer_count,
        "g0_count": g0_count,
        "g1_count": g1_count,
        "total_moves": total_moves,
        "extrusion_ratio": extrusion_ratio,
        "x_min": x_min, "x_max": x_max, "x_mean": x_mean, "x_std": x_std,
        "x_range": x_max - x_min,
        "y_min": y_min, "y_max": y_max, "y_mean": y_mean, "y_std": y_std,
        "y_range": y_max - y_min,
        "z_min": z_min, "z_max": z_max, "z_mean": z_mean, "z_std": z_std,
        "z_range": z_max - z_min,
        "e_min": e_min, "e_max": e_max, "e_mean": e_mean, "e_std": e_std,
        "f_min": f_min, "f_max": f_max, "f_mean": f_mean, "f_std": f_std,
    }
    return features

# Test on first sample
test_features = extract_gcode_features(df["gcode_text"].iloc[0])
print(f"Feature count: {len(test_features)}")
for k, v in test_features.items():
    print(f"  {k}: {v:.4f}")


## 4. Build Feature Matrix


In [ ]:
print("Extracting g-code features...")
all_features = []
valid_indices = []

for idx, row in tqdm(df.iterrows(), total=len(df)):
    try:
        feats = extract_gcode_features(row["gcode_text"])
        all_features.append(feats)
        valid_indices.append(idx)
    except Exception as e:
        print(f"Error processing {row['filename']}: {e}")

df = df.loc[valid_indices].reset_index(drop=True)
feature_df = pd.DataFrame(all_features)
print(f"\nExtracted features for {len(feature_df)} files")
print(f"Feature columns ({len(feature_df.columns)}): {list(feature_df.columns)}")


## 5. Train/Val Split (Group-Aware)

Split by `part` so that variants of the same 3D model stay together.
This prevents data leakage and forces the model to generalise to **unseen geometries**.


In [ ]:
X = feature_df.values.astype(np.float32)
X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)
y = df["label"].values
groups = df["part"].values

gss = GroupShuffleSplit(n_splits=1, test_size=VAL_SPLIT, random_state=RANDOM_SEED)
train_idx, val_idx = next(gss.split(X, y, groups))

X_train, X_val = X[train_idx], X[val_idx]
y_train, y_val = y[train_idx], y[val_idx]

# Verify no leakage
train_parts = set(groups[train_idx])
val_parts = set(groups[val_idx])
overlap = train_parts & val_parts
print(f"Train: {len(y_train)} samples, {len(train_parts)} unique parts")
print(f"Val:   {len(y_val)} samples, {len(val_parts)} unique parts")
print(f"Part overlap: {len(overlap)} (should be 0)")
print(f"\nTrain labels: pos={y_train.sum()}, neg={len(y_train)-y_train.sum()}")
print(f"Val labels:   pos={y_val.sum()}, neg={len(y_val)-y_val.sum()}")


## 6. Cross-Validation (Group-Aware)

Run 5-fold group-stratified CV to get a reliable estimate of performance
before training the final model.


In [ ]:
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)

cv_model = xgb.XGBClassifier(**XGB_PARAMS)
cv_scores = cross_val_score(
    cv_model, X_train, y_train, cv=sgkf, groups=groups[train_idx], scoring="roc_auc"
)

print(f"5-Fold Group-Stratified CV ROC-AUC:")
for i, s in enumerate(cv_scores):
    print(f"  Fold {i+1}: {s:.4f}")
print(f"  Mean:  {cv_scores.mean():.4f} +/- {cv_scores.std():.4f}")

if cv_scores.mean() > 0.95:
    print("\n⚠️  CV score is suspiciously high. Check for feature leakage.")
elif cv_scores.mean() < 0.70:
    print("\n⚠️  CV score is low. Features may not be discriminative enough — consider adding more.")
else:
    print("\n✓ CV score looks reasonable for a well-regularised model.")


## 7. Train Final Model


In [ ]:
# Scale factor for imbalanced classes
n_pos = y_train.sum()
n_neg = len(y_train) - n_pos
scale_pos_weight = n_neg / max(n_pos, 1)
print(f"scale_pos_weight: {scale_pos_weight:.2f}")

final_params = {**XGB_PARAMS, 'scale_pos_weight': scale_pos_weight}
model = xgb.XGBClassifier(**final_params)

model.fit(
    X_train, y_train,
    eval_set=[(X_train, y_train), (X_val, y_val)],
    verbose=10,
)

results = model.evals_result()
print(f"\nFinal train logloss: {results['validation_0']['logloss'][-1]:.4f}")
print(f"Final val logloss:   {results['validation_1']['logloss'][-1]:.4f}")


## 8. Evaluation


In [ ]:
# Training curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

epochs = range(len(results['validation_0']['logloss']))
axes[0].plot(epochs, results['validation_0']['logloss'], label='Train')
axes[0].plot(epochs, results['validation_1']['logloss'], label='Val')
axes[0].set_xlabel('Boosting Round')
axes[0].set_ylabel('Log Loss')
axes[0].set_title('Training Curves')
axes[0].legend()

# Overfitting gap
train_ll = np.array(results['validation_0']['logloss'])
val_ll = np.array(results['validation_1']['logloss'])
axes[1].plot(epochs, val_ll - train_ll, color='red')
axes[1].axhline(y=0, color='k', linestyle='--', alpha=0.3)
axes[1].set_xlabel('Boosting Round')
axes[1].set_ylabel('Val - Train Loss')
axes[1].set_title('Overfitting Gap')

plt.tight_layout()
plt.savefig('xgb_training_curves.png', dpi=150, bbox_inches='tight')
print('Saved xgb_training_curves.png')
plt.show()


In [ ]:
# Predictions
y_pred = model.predict(X_val)
y_probs = model.predict_proba(X_val)[:, 1]

print("Classification Report:")
print(classification_report(y_val, y_pred, target_names=["non-gun", "gun"]))

if len(np.unique(y_val)) > 1:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # Confusion matrix
    cm = confusion_matrix(y_val, y_pred)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=["non-gun", "gun"], yticklabels=["non-gun", "gun"], ax=axes[0])
    axes[0].set_xlabel("Predicted")
    axes[0].set_ylabel("Actual")
    axes[0].set_title("Confusion Matrix")

    # Confidence distribution
    gun_probs = y_probs[y_val == 1]
    nongun_probs = y_probs[y_val == 0]
    axes[1].hist(gun_probs, bins=30, alpha=0.6, label='Gun (actual)', color='red')
    axes[1].hist(nongun_probs, bins=30, alpha=0.6, label='Non-gun (actual)', color='blue')
    axes[1].set_xlabel('Predicted probability (gun)')
    axes[1].set_ylabel('Count')
    axes[1].set_title('Confidence Distribution')
    axes[1].legend()
    axes[1].axvline(x=0.5, color='k', linestyle='--', alpha=0.3)

    # Calibration curve
    fraction_pos, mean_predicted = calibration_curve(y_val, y_probs, n_bins=10, strategy='uniform')
    axes[2].plot(mean_predicted, fraction_pos, 's-', label='XGBoost', color='blue')
    axes[2].plot([0, 1], [0, 1], 'k--', label='Perfect')
    axes[2].set_xlabel('Mean predicted probability')
    axes[2].set_ylabel('Fraction of positives')
    axes[2].set_title('Calibration Curve')
    axes[2].legend()

    plt.tight_layout()
    plt.savefig('xgb_evaluation.png', dpi=150, bbox_inches='tight')
    print('Saved xgb_evaluation.png')
    plt.show()

    print(f"\nROC-AUC: {roc_auc_score(y_val, y_probs):.4f}")

    tn, fp, fn, tp = cm.ravel()
    print(f"True Positives:  {tp} (gun correctly identified)")
    print(f"True Negatives:  {tn} (non-gun correctly identified)")
    print(f"False Positives: {fp} (non-gun misclassified as gun)")
    print(f"False Negatives: {fn} (gun misclassified as non-gun)")

    # Overconfidence check
    high_conf_wrong = ((y_probs > 0.9) & (y_val == 0)).sum() + ((y_probs < 0.1) & (y_val == 1)).sum()
    print(f"\nOverconfident & wrong: {high_conf_wrong} (prob>0.9 but non-gun, or prob<0.1 but gun)")


## 9. Feature Importance


In [ ]:
# Built-in XGBoost feature importance
feature_names = list(feature_df.columns)
importances = model.feature_importances_
sorted_idx = np.argsort(importances)

fig, ax = plt.subplots(figsize=(10, 8))
ax.barh(range(len(sorted_idx)), importances[sorted_idx])
ax.set_yticks(range(len(sorted_idx)))
ax.set_yticklabels([feature_names[i] for i in sorted_idx])
ax.set_xlabel('Feature Importance (gain)')
ax.set_title('XGBoost Feature Importance')
plt.tight_layout()
plt.savefig('xgb_feature_importance.png', dpi=150, bbox_inches='tight')
print('Saved xgb_feature_importance.png')
plt.show()

print('\nTop 10 features:')
for i in sorted_idx[-10:][::-1]:
    print(f'  {feature_names[i]:20s}: {importances[i]:.4f}')


In [ ]:
# SHAP analysis (more reliable than built-in importance)
import shap

explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_val)

fig, ax = plt.subplots(figsize=(10, 8))
shap.summary_plot(shap_values, X_val, feature_names=feature_names, show=False)
plt.tight_layout()
plt.savefig('xgb_shap.png', dpi=150, bbox_inches='tight')
print('Saved xgb_shap.png')
plt.show()


## 10. Inference Helper

Classify a new g-code file.


In [ ]:
def classify_gcode(gcode_input: str, model, feature_columns, threshold=0.5):
    """Classify g-code. Accepts raw text or a filepath. Returns (is_gun: bool, confidence: float)."""
    if os.path.isfile(gcode_input):
        with open(gcode_input, "r", errors="ignore") as f:
            gcode_text = f.read()
    else:
        gcode_text = gcode_input

    features = extract_gcode_features(gcode_text)
    feat_vec = np.array([[features[c] for c in feature_columns]], dtype=np.float32)
    feat_vec = np.nan_to_num(feat_vec, nan=0.0, posinf=0.0, neginf=0.0)

    prob = model.predict_proba(feat_vec)[0, 1]
    return prob >= threshold, float(prob)


# Test inference
feature_columns = list(feature_df.columns)
is_gun, confidence = classify_gcode('output.gcode', model, feature_columns)
print(f"File: output.gcode")
print(f"Prediction: {'GUN PART' if is_gun else 'NOT gun part'} (confidence: {confidence:.4f})")


## 11. Save Model & Artifacts


In [ ]:
SAVE_DIR = Path("./saved_model_xgb")
SAVE_DIR.mkdir(exist_ok=True)

# Save XGBoost model
model.save_model(SAVE_DIR / "xgb_classifier.json")

# Save config
config = {
    "model_type": "xgboost",
    "feature_columns": list(feature_df.columns),
    "n_features": len(feature_df.columns),
    "xgb_params": XGB_PARAMS,
}
with open(SAVE_DIR / "config.json", "w") as f:
    json.dump(config, f, indent=2)

print(f"Model saved to {SAVE_DIR}")
print(f"Files: {list(SAVE_DIR.iterdir())}")
print(f"\nModel size: {(SAVE_DIR / 'xgb_classifier.json').stat().st_size / 1024:.1f} KB")
print("(Compare this to the ~2.4 GB embedding model!)")


---

## Notes

**Why this works better than the embedding model for ~2k samples:**
- XGBoost can't memorise raw g-code sequences — it only sees 28 numbers
- Regularisation is built into the algorithm (max_depth, subsample, min_child_weight)
- Group-aware splitting forces generalisation to unseen parts
- If the model fails, SHAP tells you exactly *why* — you can fix the features

**If results aren't good enough:**
1. Add more features (aspect ratios, travel distance distributions, retraction patterns)
2. Add more diverse non-gun g-code files
3. Try `max_depth=3` for even less overfitting
4. Use Optuna for hyperparameter search
